# 16 · VaR Forecast AutoML

Train Prophet + pmdarima auto_arima per trading-book VaR forecast models. Evaluates both on a holdout window, keeps the lower-SMAPE champion per book, wraps all champions in a single UC-registered pyfunc (`var_forecast_model @champion`), and writes `gold_var_forecast`.

> **Why not `databricks.automl.forecast()`?**  
> The Python API (`databricks.automl.forecast()`) is documented as [classic compute only](https://docs.databricks.com/aws/en/machine-learning/automl/forecasting-train-api) — it requires Databricks Runtime ML and cannot run on serverless compute. The newer *Serverless Forecasting* experience (Mosaic AI, Dec 2024) is [UI-only](https://docs.databricks.com/aws/en/machine-learning/train-model/serverless-forecasting/) with no Python API. The `ai_forecast()` SQL function works on serverless but only from a SQL warehouse, not a notebook.  
> This notebook implements the same Prophet + ARIMA algorithm sweep directly, which runs correctly on serverless compute.

In [0]:
%pip install -q prophet>=1.1 pmdarima>=2.0

In [0]:
import sys, os, warnings, pickle, tempfile, logging
warnings.filterwarnings("ignore")

# Silence the repeated WARNING: "getContext().extraContext() is not a CommandContext accessor"
# Root cause: MLflow's tracking context resolver calls the Databricks Java CommandContext API
# to auto-tag runs with notebook metadata. In interactive execution (dataScience_runAsset),
# the full job context isn't initialised, so the accessor is missing. In a proper notebook_task
# scheduled run this warning will not appear. We suppress it defensively here via three layers:
#   1. setLevel(CRITICAL) on the specific logger
#   2. propagate=False so parent-logger handlers don't catch it either
#   3. A root-level filter that drops messages containing 'extraContext'
_ctx_log = logging.getLogger("mlflow.tracking.context.registry")
_ctx_log.setLevel(logging.CRITICAL)
_ctx_log.propagate = False

class _NoExtraCtx(logging.Filter):
    def filter(self, record):
        return "extraContext" not in record.getMessage()
logging.getLogger().addFilter(_NoExtraCtx())

# Resolve src/ so riskcore is importable in notebook_task execution context
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb  = _ctx.notebookPath().get()          # e.g. /Users/.../src/notebooks/16_var_forecast_automl
_SRC = "/Workspace" + os.path.dirname(os.path.dirname(_nb))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from pyspark.sql import functions as F

from riskcore.config import cfg_from_widgets, widget
from riskcore.evidence import Evidence
print(f"Imports OK | riskcore from: {_SRC}")

In [0]:
cfg      = cfg_from_widgets(dbutils)
ev       = Evidence("16_var_forecast_automl")
horizon  = int(widget(dbutils, "horizon_days", "10"))
MIN_ROWS = 8   # minimum history per book; holdout is sized by holdout_size() so training keeps most rows
print(f"catalog={cfg.catalog}  schema={cfg.schema}  horizon={horizon}d  MIN_ROWS={MIN_ROWS}")

In [0]:
# gold_book_var has one row per (as_of_date, book_id, time_step) on the simulation time grid.
# time_step 1 is the 1-day horizon (same convention as the dashboard), giving one row per book per date.
var_df = spark.sql(
    f"SELECT as_of_date, book_id, var_95, expected_shortfall_95 "
    f"FROM {cfg.t('gold_book_var')} WHERE time_step = 1"
).toPandas()
assert not var_df.duplicated(["as_of_date", "book_id"]).any(), "expected one row per book per date"
var_df["as_of_date"] = pd.to_datetime(var_df["as_of_date"])
var_df = var_df.sort_values(["book_id", "as_of_date"]).reset_index(drop=True)

total = (
    var_df.groupby("as_of_date")[["var_95", "expected_shortfall_95"]]
    .sum().reset_index().assign(book_id="TOTAL")
)
var_df  = pd.concat([var_df, total], ignore_index=True)
n_dates = var_df["as_of_date"].nunique()
n_books = var_df["book_id"].nunique()
print(f"Loaded {len(var_df)} rows | {n_dates} dates, {n_books} books (inc. TOTAL)")
ev.metric("n_dates", n_dates)
ev.metric("n_books", n_books)
display(var_df[var_df["book_id"] == "TOTAL"].tail(5))

In [0]:
def smape(actual, predicted):
    return float(np.mean(
        2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted) + 1e-9)
    ))


def trial_prophet(dates, values, n_test):
    """Fit Prophet; return (smape_holdout, fitted_model)."""
    from prophet import Prophet
    ts       = pd.DataFrame({"ds": dates.values, "y": values.values})
    train_df = ts.iloc[:-n_test]
    test_df  = ts.iloc[-n_test:]
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=(len(train_df) >= 365),
        uncertainty_samples=0,
    )
    m.fit(train_df)
    future = m.make_future_dataframe(periods=n_test, freq="B", include_history=False)
    fc     = m.predict(future)
    preds  = np.maximum(fc["yhat"].values[:n_test], 0.0)
    return smape(test_df["y"].values, preds), m


def trial_arima(values, n_test):
    """Fit pmdarima auto_arima; return (smape_holdout, fitted_model)."""
    import pmdarima as pm
    arr     = values.values
    train_y = arr[:-n_test]
    test_y  = arr[-n_test:]
    model   = pm.auto_arima(
        train_y, seasonal=False, information_criterion="aic",
        stepwise=True, suppress_warnings=True, error_action="ignore",
        max_p=5, max_q=5, max_d=2,
    )
    preds = np.maximum(model.predict(n_periods=n_test), 0.0)
    return smape(test_y, preds), model

def fit_full(model_type, dates, values):
    """Refit the selected algorithm on the FULL series so forecasts start after last_ds."""
    if model_type == "prophet":
        from prophet import Prophet
        m = Prophet(daily_seasonality=False, weekly_seasonality=True,
                    yearly_seasonality=(len(values) >= 365), uncertainty_samples=0)
        m.fit(pd.DataFrame({"ds": dates.values, "y": values.values}))
        return m
    import pmdarima as pm
    return pm.auto_arima(values.values, seasonal=False, information_criterion="aic",
                         stepwise=True, suppress_warnings=True, error_action="ignore",
                         max_p=5, max_q=5, max_d=2)


def holdout_size(n_rows, horizon):
    """Holdout = horizon when history allows, else ~25% of rows (min 2) so training keeps most data."""
    return min(horizon, max(2, n_rows // 4))

print("Helpers defined: smape, trial_prophet, trial_arima, fit_full, holdout_size")

In [0]:
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/brian.mcconnell@databricks.com/{cfg.catalog}-var-forecast-automl")
MODEL_NAME = f"{cfg.catalog}.{cfg.schema}.var_forecast_model"
N_TEST     = horizon

book_champions = {}
trial_rows     = []
naive_books    = []

with mlflow.start_run(run_name="var_forecast_automl") as parent_run:
    mlflow.log_params({
        "horizon_days": horizon, "n_books": n_books, "n_dates": n_dates,
        "n_test": N_TEST, "algorithms": "prophet,arima",
    })

    for book_id in sorted(var_df["book_id"].unique()):
        g = (
            var_df[var_df["book_id"] == book_id]
            .sort_values("as_of_date")
            .reset_index(drop=True)
        )
        if len(g) < MIN_ROWS:
            print(f"  {book_id}: {len(g)} rows < {MIN_ROWS} -- skipped")
            continue

        dates  = g["as_of_date"]
        values = g["var_95"]
        N_TEST = holdout_size(len(g), horizon)
        candidates = {}

        with mlflow.start_run(run_name=f"prophet_{book_id}", nested=True):
            try:
                s, m = trial_prophet(dates, values, N_TEST)
                mlflow.log_params({"book_id": book_id, "algorithm": "prophet"})
                mlflow.log_metric("smape", s)
                candidates["prophet"] = (s, m)
                print(f"  {book_id}  prophet   SMAPE={s:.4f}")
            except Exception as exc:
                print(f"  {book_id}  prophet   ERROR: {exc}")

        with mlflow.start_run(run_name=f"arima_{book_id}", nested=True):
            try:
                s, m = trial_arima(values, N_TEST)
                mlflow.log_params({"book_id": book_id, "algorithm": "arima"})
                mlflow.log_metric("smape", s)
                candidates["arima"] = (s, m)
                print(f"  {book_id}  arima     SMAPE={s:.4f}")
            except Exception as exc:
                print(f"  {book_id}  arima     ERROR: {exc}")

        if not candidates:
            continue

        best_type     = min(candidates, key=lambda k: candidates[k][0])
        best_smape, _ = candidates[best_type]
        best_m        = fit_full(best_type, dates, values)   # refit on all rows so forecasts start after last_ds

        # Validate the refit champion; very short series can give degenerate fits that predict NaN.
        try:
            if best_type == "prophet":
                _fd  = pd.bdate_range(dates.max() + pd.Timedelta(days=1), periods=horizon)
                _chk = best_m.predict(pd.DataFrame({"ds": _fd}))["yhat"].values
            else:
                _chk = np.asarray(best_m.predict(n_periods=horizon))
            _ok = bool(np.all(np.isfinite(_chk)))
        except Exception:
            _ok = False
        if not _ok:
            print(f"  {book_id}  {best_type} refit not finite -> naive last-value fallback")
            best_type, best_m = "naive", float(values.iloc[-1])
            naive_books.append(book_id)
        book_champions[book_id] = {
            "model_type": best_type,
            "model":      best_m,
            "smape":      best_smape,
            "last_ds":    dates.max(),
        }
        trial_rows.append({"book_id": book_id, "winner": best_type, "smape": best_smape})
        print(f"  {book_id}  -> {best_type}  (SMAPE={best_smape:.4f})")

    if trial_rows:
        agg        = pd.DataFrame(trial_rows)
        mean_smape = float(agg["smape"].mean())
        prophet_w  = int((agg["winner"] == "prophet").sum())
        arima_w    = int((agg["winner"] == "arima").sum())
        mlflow.log_metrics({"mean_smape": mean_smape,
                            "prophet_wins": prophet_w, "arima_wins": arima_w})
        ev.metric("mean_smape",   round(mean_smape, 4))
        ev.metric("prophet_wins", prophet_w)
        ev.metric("arima_wins",   arima_w)
        print(f"\nSummary: mean SMAPE={mean_smape:.4f} | Prophet={prophet_w} ARIMA={arima_w}")

    mlflow.log_metric("naive_fallbacks", len(naive_books))
    ev.metric("naive_fallbacks", len(naive_books))
    parent_run_id = parent_run.info.run_id

print(f"Champions trained for {len(book_champions)} / {n_books} books")
display(pd.DataFrame(trial_rows).sort_values("smape"))

In [0]:
class ForecastModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import pickle
        with open(context.artifacts["champions_pkl"], "rb") as f:
            self._champs = pickle.load(f)

    def predict(self, context, model_input):
        import numpy as np, pandas as pd
        rows = []
        for _, req in model_input.iterrows():
            book_id      = req["book_id"]
            h            = int(req["horizon_days"])
            last_ds      = pd.Timestamp(req["last_ds"])
            future_dates = pd.bdate_range(last_ds + pd.Timedelta(days=1), periods=h)
            champ        = self._champs.get(book_id)
            if champ is None:
                preds = np.zeros(h)
            elif champ["model_type"] == "naive":
                preds = np.full(h, max(float(champ["model"]), 0.0))
            elif champ["model_type"] == "prophet":
                future_df = pd.DataFrame({"ds": future_dates})
                fc    = champ["model"].predict(future_df)
                preds = np.maximum(fc["yhat"].values[:h], 0.0)
            else:
                preds = np.maximum(champ["model"].predict(n_periods=h), 0.0)
            for i, (d, p) in enumerate(zip(future_dates, preds)):
                rows.append({
                    "book_id":         book_id,
                    "horizon_days":    i + 1,
                    "as_of_date":      d.date(),
                    "var_95_forecast": float(p),
                })
        return pd.DataFrame(rows)


with mlflow.start_run(run_id=parent_run_id):
    with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tf:
        pickle.dump(book_champions, tf)
        pkl_path = tf.name
    mlflow.pyfunc.log_model(
        artifact_path="forecast_pyfunc",
        python_model=ForecastModel(),
        artifacts={"champions_pkl": pkl_path},
        registered_model_name=MODEL_NAME,
        input_example=pd.DataFrame([{
            "book_id":      "TOTAL",
            "horizon_days": horizon,
            "last_ds":      str(pd.Timestamp.today().date()),
        }]),
    )

client   = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest   = max(int(v.version) for v in versions)
client.set_registered_model_alias(MODEL_NAME, "champion", str(latest))
print(f"Registered {MODEL_NAME} version {latest} -> @champion")
ev.metric("model_version", latest)
ev.metric("run_id",        parent_run_id)

In [0]:
if not book_champions:
    raise RuntimeError("No books had sufficient data to train; check MIN_ROWS vs available history.")

forecast_date = pd.Timestamp.today().normalize()
# One request row per book (horizon_days=full horizon); the pyfunc generates
# horizon individual step rows per book, giving n_books x horizon total rows.
req_rows = [
    {"book_id": bid, "horizon_days": horizon,
     "last_ds": str(book_champions[bid]["last_ds"].date())}
    for bid in sorted(book_champions)
]
req_df       = pd.DataFrame(req_rows)
pyfunc_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
fc_df        = pyfunc_model.predict(req_df)

fc_df["forecast_date"] = forecast_date.date()
fc_df["model_version"] = latest
fc_df["run_id"]        = parent_run_id
fc_df["created_ts"]    = pd.Timestamp.utcnow()

fcast_sdf = spark.createDataFrame(fc_df)
(
    fcast_sdf.write.mode("overwrite")
    .option("replaceWhere", f"forecast_date = '{forecast_date.date()}'")
    .saveAsTable(cfg.t("gold_var_forecast"))
)
n_rows = fcast_sdf.count()
print(f"gold_var_forecast: wrote {n_rows} rows for {forecast_date.date()}, "
      f"{len(book_champions)} books x {horizon} horizon steps")
ev.metric("forecast_rows", n_rows)
ev.metric("books_covered", len(book_champions))
display(fcast_sdf.filter("book_id = 'TOTAL'"))

In [0]:
ev.save()